# Module 14: Sensitivity Analysis and Partial Identification

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

When an assumption cannot be defended, two things remain. **Sensitivity
analysis** asks how badly it would have to fail. **Partial identification**
drops it and reports what the data alone can bound.

This module runs three of them on the same estimate and gets three different
verdicts, including one that is uncomfortable and one that fails outright.

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

In [ ]:
d["tr"] = d["agency_id"].isin(KEEP).astype(float)
d["yrc"] = d["yr"] - d["yr"].min()
d["settled"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "phase")).astype(float)
e0, l0, h0, _ = fit(d, KEEP)
print(f"  the estimate as reported: {e0:+.2f}%  [{l0:+.1f}, {h0:+.1f}]")

## 2. Sensitivity to an unmeasured trend

Break parallel trends by a known amount and find the breakdown point.

In [ ]:
rows = []
for delta in [0.0, -1.0, -2.0, -3.0, -4.0]:
    s = d.copy()
    s["adj"] = np.log(s["n_arrests"]) + np.log(1 + delta / 100) * s["tr"] * s["yrc"]
    e, lo, hi, _ = fit(s, KEEP, offset=s["adj"])
    rows.append({"hidden trend": f"{delta:+.1f}% a year", "estimate": f"{e:+.1f}%",
                 "interval": f"[{lo:+.1f}, {hi:+.1f}]",
                 "excludes zero": "yes" if not (lo < 0 < hi) else "no"})
pre = d[d["period"] == "before"]
z = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", pre,
            family=sm.families.Poisson(), offset=pre["lo"]).fit()
k = [x for x in z.params.index if "yr" in x and "tr" in x][0]
lo, hi = [pct(v) for v in z.conf_int().loc[k]]
print(f"  the pre period puts the trend difference at "
      f"{pct(z.params[k]):+.2f}% a year, interval [{lo:+.2f}, {hi:+.2f}]\n")
pd.DataFrame(rows).set_index("hidden trend")

It takes about **3.4 percent a year** to erase the estimate, and the pre
period's interval reaches **3.31**.

**The breakdown point sits just inside what the data cannot rule out.** That
is a real caveat and it should be reported as one rather than as robustness.

## 3. Partial identification: what the data bounds without the assumption

Drop parallel trends entirely. The treated group's Y(1) after the program is
observed; Y(0) is not. Manski's approach is to bound Y(0) by whatever the data
and a minimal assumption allow, and report the resulting interval.

In [ ]:
tr = d[d["agency_id"].isin(KEEP)]
y1 = rate(tr[tr["period"] == "after"])
before = rate(tr[tr["period"] == "before"])
after_c = {a: rate(d[(d["agency_id"] == a) & (d["period"] == "after")])
           for a in COMPARISON}

print(f"  Y(1) for the treated group after the program: {y1:.3f}")
print(f"  the untreated agencies after the program span "
      f"[{min(after_c.values()):.3f}, {max(after_c.values()):.3f}]\n")
lo_b = 100 * (y1 / max(after_c.values()) - 1)
hi_b = 100 * (y1 / min(after_c.values()) - 1)
print(f"  bound A, Y(0) somewhere in that span:")
print(f"    the effect lies in [{lo_b:+.1f}%, {hi_b:+.1f}%]")
lo_c = 100 * (y1 / before - 1)
print(f"\n  bound B, Y(0) no higher than the treated group's own before value:")
print(f"    the effect lies in [{lo_c:+.1f}%, {hi_b:+.1f}%]")
print(f"\n  the truth: {TRUTH:+.1f}%")

**Bound A excludes the truth.** It is not a valid bound here, and the reason
is the overlap failure from [Module 10](Module_10_Propensity_Scores.ipynb):
the treated agencies' Y(0) is above every untreated agency's observed rate, so
the range of untreated outcomes is not a range Y(0) lives in.

**A worst case bound built from the wrong support is not conservative. It is
wrong.** Reporting it as "we make no assumptions" would be the most misleading
claim in this series.

**Bound B is valid** and uses one weak assumption: the treated group's Y(0)
did not rise above its own pre program level. It contains the truth and it is
**75 percentage points wide.**

That is the honest trade partial identification offers here: a bound that
assumes almost nothing and rules out almost nothing.

## 4. How much unobserved selection would be needed

A third angle, in the spirit of Oster: compare how much the estimate moved
when observed controls were added, and ask how much further unobservables
would have to move it.

In [ ]:
short = smf.glm("n_uof ~ settled+phase", d, family=sm.families.Poisson(),
                offset=d["lo"]).fit()
full = smf.glm("n_uof ~ C(agency_id)+C(year_month)+settled+phase", d,
               family=sm.families.Poisson(), offset=d["lo"]).fit()
bs, bf = pct(short.params["settled"]), pct(full.params["settled"])
print(f"  with no controls:    {bs:+.2f}%")
print(f"  with full controls:  {bf:+.2f}%")
print(f"  the observed controls moved it {bf - bs:+.2f} points")
print(f"\n  to reach zero, unobservables would have to move it a further "
      f"{0 - bf:+.2f} points,")
print(f"  which is {abs((0 - bf) / (bf - bs)):.2f} times the pull of everything observed")

Unobservables would have to be **3.3 times as influential as every observed
control combined.** Conventionally a ratio above 1 is taken as reassuring.

**Two caveats, both worth stating.** Oster's statistic is defined for least
squares with R squared, and this is a Poisson model, so the ratio here is an
analogy rather than the published statistic. And the logic assumes the
unobservables resemble the observables in how they relate to treatment, which
is an assumption, not a fact.

## 5. Three analyses, three verdicts

| Analysis | Verdict |
|---|---|
| Unmeasured trend | **uncomfortable**: the breakdown point is inside the pre period's interval |
| Partial identification | **uninformative**: 75 points wide, and the naive version is invalid |
| Unobserved selection | **reassuring**: 3.3 times the observed pull would be needed |

**Report all three.** Reporting only the third would be a selective robustness
claim, and reporting only the first would understate what the design achieved.

## Exercise

Rambachan and Roth propose bounding the post period violation by a multiple of
the largest pre period violation observed. Implement the simplest version.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    pre2 = d[d["period"] == "before"].copy()
    pre2["half"] = (pre2["year_month"] >= "2021-04").astype(float)
    pre2["trh"] = pre2["tr"] * pre2["half"]
    zp = smf.glm("n_uof ~ C(agency_id)+C(year_month)+trh", pre2,
                 family=sm.families.Poisson(), offset=pre2["lo"]).fit()
    worst_pre = abs(pct(zp.params["trh"]))
    print(f"  largest violation visible in the pre period: {worst_pre:.2f} percent\n")
    rows = []
    for M in [0.0, 1.0, 2.0, 3.0]:
        rows.append({"post period violation allowed":
                         f"{M:.0f} times the pre period one ({M * worst_pre:.1f}%)",
                     "the effect could be as small as":
                         f"{e0 + M * worst_pre:+.1f}%",
                     "still a reduction": "yes" if e0 + M * worst_pre < 0 else "NO"})
    display(pd.DataFrame(rows).set_index("post period violation allowed"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The conclusion that the program reduced use of force survives a post period
violation of one or two times the largest one visible in the pre period, and
does not survive three times.

**That sentence is the useful output of a sensitivity analysis**, and it is
the form Rambachan and Roth argue for: not "the assumption holds" and not "the
assumption might fail", but **the conclusion holds as long as the violation is
no more than M times what the pre period shows.** A reader can then supply
their own judgment about M.

The version implemented here is the crudest one, a single relative magnitude
bound on an average post period violation. The published method builds the
bound period by period and inverts it to get a confidence set, which needs
more machinery than this module carries and gives a similar qualitative
answer on data this thin.

</details>

---

**Next:** [Module 15: Heterogeneous Effects, and the Temptation to Find Them](Module_15_Heterogeneous_Effects.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*